In [86]:
import os
import re
import sys
import joblib
import pandas as pd
import numpy as np
import clingo

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from api.data_utils import extract_from_json, aggregate_to_operator_day, generate_clingo_facts, generate_physical_instance

In [87]:
def parse_assignments_to_df(raw_assignments):
    """
    Prende la lista di stringhe di Clingo e la trasforma in un DataFrame
    con colonne ['Patient', 'Operator']
    """
    parsed_data = []
    # Cerca la parola assignment, poi cattura il primo numero (Operatore) e il secondo (Paziente)
    pattern = re.compile(r"assignment\((-?\d+),\s*(\d+)")
    
    for stringa in raw_assignments:
        match = pattern.search(stringa)
        if match:
            op_id = int(match.group(1))
            pat_id = int(match.group(2))
            parsed_data.append({"Patient": pat_id, "Operator": op_id})
            
    return pd.DataFrame(parsed_data)

In [88]:
def analyze_agenda_shifts(raw_base, raw_ns):
    """
    Confronta gli assegnamenti e restituisce un DataFrame con l'analisi degli scostamenti.
    """
    # 1. Parsiamo le due liste
    df_base = parse_assignments_to_df(raw_base)
    df_ns = parse_assignments_to_df(raw_ns)
    
    # 2. Uniamo i dati basandoci sull'ID del Paziente
    # Usiamo 'outer' nel caso remoto in cui un paziente esista solo in un'agenda
    df_compare = pd.merge(df_base, df_ns, on="Patient", suffixes=('_Base', '_NS'), how="outer")
    df_compare = df_compare.fillna(-999).astype(int) # -999 per eventuali dati mancanti
    
    # 3. Classifichiamo cosa è successo ad ogni paziente
    def categorize_shift(row):
        op_base = row['Operator_Base']
        op_ns = row['Operator_NS']
        
        if op_base == op_ns:
            if op_base == -1:
                return "Non Assegnato in Entrambi"
            return "Invariato"
        elif op_base != -1 and op_ns == -1:
            return "Perso dal ML (Aggiunto a -1)"
        elif op_base == -1 and op_ns != -1:
            return "Salvato dal ML (Tolto da -1)"
        else:
            return "Spostato su nuovo Operatore"

    df_compare['Status'] = df_compare.apply(categorize_shift, axis=1)
    
    # Riordiniamo le colonne per bellezza
    df_compare = df_compare[['Patient', 'Status', 'Operator_Base', 'Operator_NS']]
    
    return df_compare

In [89]:
def generate_predictions_for_date(json_path, target_date, train_columns, trained_models, output_filename):
    df_raw = extract_from_json([json_path])
    df_agg = aggregate_to_operator_day(df_raw)
    
    df_today = df_agg[df_agg['planning_date'].astype(str).str.contains(target_date)].copy()
    
    if df_today.empty:
        print("Nessun dato trovato per questa data.")
        return
        
    today_ids = df_today['operator_id']
    X_today = df_today.drop(columns=['operator_id', 'planning_date', 'target_assignments'], errors='ignore')
    
    cat_cols = X_today.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    X_today = pd.get_dummies(X_today, columns=cat_cols, drop_first=True, dtype=int)
    
    X_today = X_today.reindex(columns=train_columns, fill_value=0)
    
    q10 = trained_models['q10'].predict(X_today)
    q50 = trained_models['q50'].predict(X_today)
    q90 = trained_models['q90'].predict(X_today)
    
    y_dummy = np.zeros(len(today_ids))
    
    generate_clingo_facts(
        X_test=df_today,
        y_test=pd.Series(y_dummy), 
        q10=q10,
        q50=q50,
        q90=q90,
        original_ids=today_ids, 
        output_filename=output_filename
    )

In [90]:
def run_neuro_symbolic_solver(rules_files_list, facts_file, ml_file, timeout_seconds=30.0):    
    ctl = clingo.Control(["--opt-strategy=usc,k,0,4", "--opt-usc-shrink=bin"])
    
    for r_file in rules_files_list:
        ctl.load(r_file)
        
    ctl.load(facts_file)
    if ml_file:
        ctl.load(ml_file)
    
    ctl.ground([("base", [])])
    
    best_cost = None
    assignments = []
    unassigned_count = 0
    
    def on_model(model):
        nonlocal best_cost, assignments, unassigned_count
        best_cost = model.cost
        assignments = [str(sym) for sym in model.symbols(shown=True)]
        unassigned_count = sum(1 for a in assignments if "assignment(-1" in a)
    
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout_seconds)
        if not finished:
            handle.cancel()
    
    return assignments, best_cost, unassigned_count

In [ ]:
def run_monthly_experiment(json_path, start_date, end_date, train_columns, trained_models, timeout=30.0):
    dates = pd.date_range(start=start_date, end=end_date)
    results = []
    all_shifts_list = []
    
    os.makedirs("encodings/facts", exist_ok=True)
    
    for date_obj in dates:
        date_str = date_obj.strftime('%Y-%m-%d')
        print(f"GIORNO: {date_str} ", end="")
        
        try:
            generate_physical_instance(json_path, date_str, "encodings/facts/day_facts.lp")
        except Exception as e:
            print(f"[Saltato: Errore estrazione]")
            continue
            
        with open("encodings/facts/day_facts.lp", 'r') as f:
            if len(f.readlines()) < 10:
                print(f"[Saltato: Ospedale Vuoto]")
                continue
        
        try:
            generate_predictions_for_date(json_path, date_str, train_columns, trained_models, "encodings/facts/ml_facts.lp")
        except Exception as e:
            print(f"[Saltato: Errore ML]")
            continue
            
        # RUN BASELINE
        ass_base, cost_base, unass_base = run_neuro_symbolic_solver(
            rules_files_list=["encodings/base_rules.lp"],
            facts_file="encodings/facts/day_facts.lp",
            ml_file=None,
            timeout_seconds=timeout
        )
    
        # RUN NEURO-SIMBOLICO
        ass_ns, cost_ns, unass_ns = run_neuro_symbolic_solver(
            rules_files_list=["encodings/base_rules.lp", "encodings/ml_rules.lp"],
            facts_file="encodings/facts/day_facts.lp",
            ml_file="encodings/facts/ml_facts.lp",
            timeout_seconds=timeout
        )
        
        # ANALISI DEGLI SCOSTAMENTI IN LINEA
        df_analisi = analyze_agenda_shifts(ass_base, ass_ns)
        
        # Estraiamo i conteggi per questa giornata
        counts = df_analisi['Status'].value_counts()
        invariati = counts.get("Invariato", 0)
        spostati = counts.get("Spostato su nuovo Operatore", 0)
        salvati = counts.get("Salvato dal ML (Tolto da -1)", 0)
        persi = counts.get("Perso dal ML (Aggiunto a -1)", 0)
        

        df_analisi['Data'] = date_str
        all_shifts_list.append(df_analisi)
        
        cost_base_list = cost_base if cost_base is not None else [0, 0, 0]
        cost_ns_list = cost_ns if cost_ns is not None else [0, 0, 0]
        
        results.append({
            'Data': date_str,
            'Pazienti_A_Terra_Baseline': unass_base,
            'Pazienti_A_Terra_NS': unass_ns,
            'Assegnazioni_Totali_Baseline': len(ass_base) - unass_base,
            'Assegnazioni_Totali_NS': len(ass_ns) - unass_ns,
            'Costo_W2_Baseline': cost_base_list[1] if len(cost_base_list) > 1 else 0,
            'Costo_W2_NS': cost_ns_list[1] if len(cost_ns_list) > 1 else 0,
            'Pazienti_Invariati': invariati,
            'Pazienti_Spostati': spostati,
            'Salvati_da_ML': salvati,
            'Persi_da_ML': persi
        })
        print(f"[OK]")

    df_results = pd.DataFrame(results)
    
    df_all_shifts = pd.concat(all_shifts_list, ignore_index=True) if all_shifts_list else pd.DataFrame()
    
    return df_results, df_all_shifts

In [94]:
def indagine_overbooking(raw_base, raw_ns, date_str, op_target):
    """
    Analizza il carico di lavoro di un operatore specifico per scovare l'overbooking.
    """
    df_base = parse_assignments_to_df(raw_base)
    carico_base = len(df_base[df_base['Operator'] == op_target])
    
    df_ns = parse_assignments_to_df(raw_ns)
    carico_ns = len(df_ns[df_ns['Operator'] == op_target])
    
    capacita_ml = "Sconosciuta"
    burden_score = "Sconosciuto"
    conf_level = "Sconosciuto"
    
    try:
        with open("encodings/facts/ml_facts.lp", 'r') as f:
            for line in f:
                # Cerca ml_capacity(31, X, Y)
                if f"ml_capacity({op_target}," in line:
                    parti = line.split(',')
                    capacita_ml = int(parti[1].strip())
                    conf_level = int(parti[2].replace(').', '').strip())
                # Cerca operator_burden(31, Z)
                if f"operator_burden({op_target}," in line:
                    burden_score = int(line.split(',')[1].replace(').', '').strip())
    except:
        pass
        
    print(f"INDAGINE OPERATORE {op_target} (Data: {date_str})")
    print(f"Assegnazioni forzate dalla Baseline : {carico_base} pazienti")
    print(f"Capacità massima sicura (Predetta)  : {capacita_ml} pazienti (Confidenza: {conf_level})")
    print(f"Burden Score (Stress Operatore)     : {burden_score}")
    print(f"Assegnazioni corrette dal ML (NS)   : {carico_ns} pazienti")
    print(f"Pazienti salvati dall'overbooking   : {carico_base - carico_ns}\n")

In [ ]:
def esegui_indagine_mirata(json_path, date_str, op_target, train_cols, models, timeout=20.0):
    """
    Ricrea l'ambiente per un singolo giorno, esegue i due solver e lancia l'indagine.
    """
    
    # 1. Rigeneriamo i file fisici e predittivi SOLO per questa data
    generate_physical_instance(json_path, date_str, "encodings/facts/day_facts.lp")
    generate_predictions_for_date(json_path, date_str, train_cols, models, "encodings/facts/ml_facts.lp")
    
    # 2. Eseguiamo il solutore Baseline
    raw_base, _, _ = run_neuro_symbolic_solver(
        rules_files_list=["encodings/base_rules.lp"],
        facts_file="encodings/facts/day_facts.lp",
        ml_file=None,
        timeout_seconds=timeout
    )
    
    # 3. Eseguiamo il solutore Neuro-Simbolico
    raw_ns, _, _ = run_neuro_symbolic_solver(
        rules_files_list=["encodings/base_rules.lp", "encodings/ml_rules.lp"],
        facts_file="encodings/facts/day_facts.lp",
        ml_file="encodings/facts/ml_facts.lp",
        timeout_seconds=timeout
    )
    
    # 4. Chiamiamo la tua funzione di indagine ora che abbiamo i dati!
    indagine_overbooking(raw_base, raw_ns, date_str, op_target)

INDAGINE OPERATORE 50 (Data: 2023-01-26)
Assegnazioni forzate dalla Baseline : 4 pazienti
Capacità massima sicura (Predetta)  : 4 pazienti (Confidenza: 1)
Burden Score (Stress Operatore)     : 11
Assegnazioni corrette dal ML (NS)   : 3 pazienti
Pazienti salvati dall'overbooking   : 1



In [ ]:
try:
    trained_models = joblib.load("saved_models/best_quantiles_model.pkl")
    train_columns = joblib.load("saved_models/train_columns.pkl")
except FileNotFoundError:
    print("File dei modelli non trovati.")

data_url = 'data/anon_boardHistory_Org9_2023-01-01_2023-01-31.json'

df_esperimento, df_dettaglio_spostamenti = run_monthly_experiment(
    json_path=data_url, 
    start_date='2023-01-01', 
    end_date='2023-01-31', 
    train_columns=train_columns, 
    trained_models=trained_models,
    timeout=20.0
)

display(df_esperimento)

print("\n\nIMPATTO MACHINE LEARNING SULL'INTERO MESE\n")
print(f"Totale Pazienti Invariati: {df_esperimento['Pazienti_Invariati'].sum()}")
print(f"Totale Pazienti Riassegnati (Spostati per bilanciare): {df_esperimento['Pazienti_Spostati'].sum()}")
print(f"Pazienti precedentemente a terra, salvati dal ML: {df_esperimento['Salvati_da_ML'].sum()}")
print(f"Pazienti persi (sacrificati per limiti strutturali): {df_esperimento['Persi_da_ML'].sum()}")


In [ ]:
giorno_target = '2023-01-26'
operatore = 50

dettaglio_giorno = df_dettaglio_spostamenti[
    (df_dettaglio_spostamenti['Data'] == giorno_target) & 
    (df_dettaglio_spostamenti['Status'] != 'Invariato')
]

print(f"Dettaglio Spostamenti per il {giorno_target}:")
display(dettaglio_giorno)

esegui_indagine_mirata(data_url, giorno_target, operatore, train_columns, trained_models)